# Week 4, day 3 (morning) — Worksheet 06 SOLUTIONS: Governance in practice — dbt and the platform

The abstractions become concrete here. The lecture claims dbt supports
governance by helping teams document models and columns, define quality tests,
track dependencies, standardise transformation logic, generate lineage, and
produce trusted analytics-ready datasets.

You built exactly such a project yesterday. Open it and check the claim.

> *Open `week4_day2_afternoon/dbt-project/demo/` alongside this sheet. You do
> not need to run anything.*

**These are discussion answers, not a marking scheme.** Where a question asks for a judgement, the reasoning matters more than matching what is written here — a group that disagrees well has done the exercise.

STEP 1 — find the governance features

### Question 1

For each of the lecture's six claims, find the **specific file** in the
project that delivers it, or say it is absent:
document models and columns · define tests · track dependencies · standardise
logic · generate lineage · trusted analytics-ready datasets.

All six are present, which is the point:

**Document models and columns** — `models/staging/stg__models.yml` carries
`description:` on models and columns; `models/staging/_stg__docs.md` holds a
reusable doc block that several columns share via `doc()`.

**Define tests** — `models/edw/edw__models.yml` has `unique`, `not_null`,
`accepted_values` and `relationships`; `tests/generic/` holds two custom tests;
`models/edw/unit_tests.yml` tests the Type 6 logic against mock rows.

**Track dependencies** — every model uses `ref()` and `source()`. That is the
dependency graph; nothing else is needed.

**Standardise logic** — `macros/` — `surrogate_key.sql`, `to_active_flag.sql`.
A rule written once and reused rather than copy-pasted.

**Generate lineage** — falls out of `ref()`. `dbt docs generate` builds it, and
`dbt ls --select +model` walks it.

**Trusted analytics-ready datasets** — the marts layer, guarded by the tests
above and declared to consumers in `models/exposures.yml`.

STEP 2 — governance as code

### Question 2

A test in a dbt project is an agreement about data, written so a machine
enforces it. Pick **two** tests from the project and write, in plain business
language, the agreement each one encodes — the sentence a business owner would
recognise.

**`relationships` on `fct_sales.product_sk` → `dim_product_t6`.** "Every
sale we report must belong to a product we know about." An orphaned sale means
either a product was deleted or the fact was built against stale dimensions —
and either way the revenue total is wrong in a way that will not look wrong.

**`accepted_values` on `dim_store.region`.** "Every store belongs to one of our
five sales regions." A new region appearing without warning means someone
changed the reference data without telling anyone — Worksheet 01's risk 2, caught
automatically.

The general point: these are business agreements, not technical checks. Their
value is that they fail the build, which converts a silent wrong number into a
loud stopped pipeline.

STEP 3 — what dbt does not give you

### Question 3

dbt covers part of governance. Name **three governance needs it does not
address**, and say what would.

dbt governs *transformation*. Outside that boundary:

**Access control.** dbt does not decide who may query the marts. That is
warehouse-side — roles and grants, which is why yesterday's lab spent a script
on ownership before dbt would run at all.

**Classification of sensitive data.** Nothing in the project says which columns
are personal or confidential. That needs tagging and classification in the
platform — Snowflake tags and masking policies, or Purview classification.

**Coverage beyond the project.** dbt's lineage stops at its own boundary. The
source systems feeding `raw`, and the dashboards downstream of the marts, are
invisible unless declared — which is what an `exposure` does, and only for what
someone remembered to declare.

Groups often add: retention and deletion, audit logging, and cost governance.
None are dbt's job.

STEP 4 — two layers in the cloud

### Question 4

The lecture splits Azure governance in two: **Azure Governance** (management
groups, RBAC, Azure Policy, monitoring, cost management, deployment guardrails)
governs *cloud resources*; **Microsoft Purview** (catalogues, lineage,
classification, access control) governs *data assets*.

Sort these into the two layers, and say which the dbt project touches:
who may query a table · which regions a database may be deployed to · what a
column means · how much the warehouse costs · which columns hold personal data ·
which dashboard breaks if a model changes.

**Resource layer** — which regions a database may be deployed to (policy);
how much the warehouse costs (cost management). These are about the
infrastructure, and dbt has no view of them.

**Data layer** — what a column means (catalogue); which columns hold personal
data (classification); which dashboard breaks if a model changes (lineage).

**Both** — who may query a table. Identity and role assignment are a resource
concern; the grant on the specific object is a data concern. This is the one
students most often place in only one layer, and the split is why yesterday's
permission error existed at all.

**What the dbt project touches:** column meaning (descriptions), and lineage
(`ref()` plus exposures). It does *not* touch classification, cost, region
policy, or identity — those are platform features that governance concepts
become real through.

STEP 5 — the honest assessment

### Question 5

Yesterday's project has documentation, tests, lineage and declared
consumers. **Would you call it governed?** Take a position, and name what you
would add first.

The defensible answer is "partly, and the missing half is the part with
legal consequences."

*What is governed:* the transformation logic. It is documented, tested,
version-controlled, reviewable, and its dependencies are explicit. Compared with
a folder of ad-hoc SQL, that is a substantial improvement — and it is the half
engineers control directly.

*What is not:* nobody is named as owner of any dataset. Nothing is classified as
sensitive. There is no access policy in the repository — the lab hands ownership
to `SYSADMIN` and moves on. There is no retention policy, and no audit of who
queried what.

*What to add first* — most groups say ownership, and that is right. Every other
control needs somebody accountable to define and maintain it. Classification is
the usual second, because it is the one where being wrong is expensive.

The honest summary: the project demonstrates governance *mechanisms* well and
governance *decisions* not at all — which is a fair description of most real
dbt projects, and worth recognising before someone calls a repository a
governance programme.